# Veritas — Phi-3 QLoRA + DPO Training (Colab T4)

**Sequence**
1. Check GPU → Install deps → Clone repo → Upload data (if needed) → Preflight → QLoRA dry-run → **QLoRA train** → Download adapter
2. New session → Install deps → Clone repo → Upload QLoRA adapter → Upload DPO data (if needed) → Preflight → DPO dry-run → **DPO train** → Download DPO adapter

**DPO cannot start without the QLoRA adapter.** The training script checks for it and exits with a skipped report if it is missing.

**These adapters affect explanation generation only.**  
The `transformer_verifier_clean` checkpoint decides the verdict label. These runs do not change it.

---
## Part 1 — QLoRA

### Cell Q1 — Verify T4 GPU

In [ ]:
import subprocess, sys

result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True
)
if result.returncode != 0:
    raise RuntimeError(
        "No GPU found. Go to: Runtime → Change runtime type → T4 GPU"
    )
print(result.stdout.strip())

import torch
assert torch.cuda.is_available(), "CUDA not available after GPU check — restart and retry"
print(f"device : {torch.cuda.get_device_name(0)}")
print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

### Cell Q2 — Install dependencies

In [ ]:
%pip install -q \
    "transformers>=4.40.0" \
    "datasets>=2.19.0" \
    "peft>=0.10.0" \
    "trl>=0.8.6" \
    "accelerate>=0.29.0" \
    "bitsandbytes>=0.43.0" \
    "pyyaml"
print("Install complete — now click Runtime → Restart session, then continue from Cell Q3.")

> **After the install finishes: Runtime → Restart session.**  
> Then skip back here and run from Cell Q3 onward. You do not need to re-run Q1 or Q2.

### Cell Q3 — Clone repo (brings scripts, configs, and dataset files)

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/sushildalavi/veritas.git"
REPO_DIR = Path("/content/veritas")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())
!git log --oneline -3

### Cell Q4 — Upload data files (skip if clone already brought them)

The repo commits `data/explanations/` so cloning normally provides these files.  
This cell uploads them only if they are missing (e.g. if you are working from a fork that does not have them).

In [ ]:
from pathlib import Path

REQUIRED_FILES = [
    "data/explanations/sft_train.jsonl",
    "data/explanations/sft_val.jsonl",
]

missing = [f for f in REQUIRED_FILES if not Path(f).exists()]

if not missing:
    print("All data files already present — skipping upload.")
    for f in REQUIRED_FILES:
        size = Path(f).stat().st_size // 1024
        print(f"  {f}  ({size} KB)")
else:
    print("Missing files — upload them now:")
    for f in missing:
        print(f"  {f}")
    from google.colab import files
    uploaded = files.upload()  # select sft_train.jsonl and sft_val.jsonl
    for name, data in uploaded.items():
        if "sft_train" in name:
            dest = Path("data/explanations/sft_train.jsonl")
        elif "sft_val" in name:
            dest = Path("data/explanations/sft_val.jsonl")
        else:
            print(f"Unexpected file: {name} — skipping")
            continue
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(data)
        print(f"  Saved {len(data)//1024} KB → {dest}")

### Cell Q5 — Preflight: validate paths and print config

In [ ]:
import yaml
from pathlib import Path

cfg_path = Path("configs/phi3_qlora.yaml")
cfg = yaml.safe_load(cfg_path.read_text())

print("=" * 55)
print("configs/phi3_qlora.yaml")
print("=" * 55)
for k, v in cfg.items():
    print(f"  {k}: {v}")
print()

checks = {
    "train_file" : cfg["train_file"],
    "eval_file"  : cfg["eval_file"],
}
all_ok = True
print("File checks:")
for label, path in checks.items():
    p = Path(path)
    if p.exists():
        lines = sum(1 for _ in p.open())
        print(f"  OK  {label}: {path}  ({lines} rows)")
    else:
        print(f"  MISSING  {label}: {path}")
        all_ok = False

Path(cfg["output_dir"]).parent.mkdir(parents=True, exist_ok=True)
print(f"  OK  output parent: {Path(cfg['output_dir']).parent}")

if not all_ok:
    raise FileNotFoundError("One or more required files are missing — re-run Cell Q4.")
print("\nPreflight passed.")

### Cell Q6 — QLoRA dry-run (non-training validation)

In [ ]:
import json
from pathlib import Path

!python3 scripts/train_phi3_qlora.py \
    --config configs/phi3_qlora.yaml \
    --dry-run \
    --report-json reports/phi3_qlora_dryrun.json \
    --report-md  reports/phi3_qlora_dryrun.md

report = json.loads(Path("reports/phi3_qlora_dryrun.json").read_text())
print("status:", report["status"])
print()
print("preflight_checks:")
all_ok = True
for c in report["preflight_checks"]:
    mark = "OK " if c["exists"] else "FAIL"
    print(f"  [{mark}] {c['name']}: {c['path']}")
    if not c["exists"]:
        all_ok = False

if not all_ok:
    raise RuntimeError("Dry-run preflight failed — fix missing paths before training.")
print("\nDry-run passed. Ready to train.")

### Cell Q7 — Run QLoRA training

Expected time: **30–50 min** on a T4.  
Do not interrupt mid-run. If the session disconnects, see recovery notes at the bottom of this notebook.

In [ ]:
!python3 scripts/train_phi3_qlora.py \
    --config configs/phi3_qlora.yaml \
    --report-json reports/phi3_qlora_training_metrics.json \
    --report-md   reports/phi3_qlora_training_metrics.md \
    --before-after-md reports/phi3_qlora_before_after_examples.md

### Cell Q8 — Verify adapter files and print training metrics

In [ ]:
import json
from pathlib import Path

report_path = Path("reports/phi3_qlora_training_metrics.json")
if not report_path.exists():
    raise FileNotFoundError("Training report missing — did Cell Q7 finish without errors?")

report = json.loads(report_path.read_text())
print("status     :", report["status"])
print("adapter_path:", report["adapter_path"])

if report["status"] == "skipped":
    raise RuntimeError("Training was skipped. Reason: " + report.get("reason", "unknown"))
if report["status"] != "trained":
    raise RuntimeError(f"Unexpected status: {report['status']}")

print("\ntrain_metrics:")
for k, v in report["train_metrics"].items():
    print(f"  {k}: {v}")

adapter_dir = Path(report["adapter_path"])
files_found = sorted(adapter_dir.iterdir())
print(f"\nadapter files ({adapter_dir}):")
for f in files_found:
    print(f"  {f.name}  ({f.stat().st_size // 1024} KB)")

has_config  = any(f.name == "adapter_config.json" for f in files_found)
has_weights = any("adapter_model" in f.name for f in files_found)
if not has_config:
    raise FileNotFoundError("adapter_config.json missing — training may not have saved correctly.")
if not has_weights:
    raise FileNotFoundError("adapter_model weights missing — training may not have saved correctly.")
print("\nAdapter complete.")

### Cell Q9 — Download QLoRA adapter and reports

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

adapter_dir = Path("adapters/phi3_veritas_qlora")
zip_path    = Path("/content/phi3_veritas_qlora.zip")

shutil.make_archive(
    str(zip_path.with_suffix("")), "zip",
    adapter_dir.parent, adapter_dir.name
)
print(f"Zipped: {zip_path}  ({zip_path.stat().st_size // 1024} KB)")
files.download(str(zip_path))

for rpt in [
    "reports/phi3_qlora_training_metrics.json",
    "reports/phi3_qlora_training_metrics.md",
    "reports/phi3_qlora_before_after_examples.md",
]:
    if Path(rpt).exists():
        files.download(rpt)
    else:
        print(f"  (missing: {rpt})")

---
## Part 2 — DPO

**Start a new Colab session** (or continue in this one if still active).  
You must have `phi3_veritas_qlora.zip` from Cell Q9 before proceeding.

### Cell D1 — Verify T4 GPU

In [ ]:
import subprocess
result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True
)
if result.returncode != 0:
    raise RuntimeError("No GPU. Runtime → Change runtime type → T4 GPU")
print(result.stdout.strip())

import torch
assert torch.cuda.is_available()
print(f"device : {torch.cuda.get_device_name(0)}")
print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

### Cell D2 — Install dependencies

In [ ]:
%pip install -q \
    "transformers>=4.40.0" \
    "datasets>=2.19.0" \
    "peft>=0.10.0" \
    "trl>=0.8.6" \
    "accelerate>=0.29.0" \
    "bitsandbytes>=0.43.0" \
    "pyyaml"
print("Done — Runtime → Restart session, then continue from Cell D3.")

### Cell D3 — Clone repo

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/sushildalavi/veritas.git"
REPO_DIR = Path("/content/veritas")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())
!git log --oneline -3

### Cell D4 — Upload QLoRA adapter (required — DPO cannot run without it)

Upload `phi3_veritas_qlora.zip` downloaded from Cell Q9.

In [ ]:
import zipfile
from pathlib import Path
from google.colab import files

adapter_dest = Path("adapters/phi3_veritas_qlora")

if adapter_dest.exists() and any(adapter_dest.iterdir()):
    print(f"QLoRA adapter already present at {adapter_dest} — skipping upload.")
    for f in sorted(adapter_dest.iterdir()):
        print(f"  {f.name}")
else:
    print("Select phi3_veritas_qlora.zip from your local machine:")
    uploaded = files.upload()

    zip_name = next(iter(uploaded))
    zip_path = Path(f"/content/{zip_name}")
    zip_path.write_bytes(uploaded[zip_name])

    # Verify the zip contains the expected top-level folder
    with zipfile.ZipFile(zip_path) as zf:
        top_level = {p.split("/")[0] for p in zf.namelist()}
        print("Top-level entries in zip:", top_level)
        if "phi3_veritas_qlora" not in top_level:
            raise ValueError(
                "Zip does not contain a top-level folder named phi3_veritas_qlora. "
                "Check you are uploading the file from Cell Q9."
            )
        Path("adapters").mkdir(exist_ok=True)
        zf.extractall("adapters")

    print(f"Extracted to {adapter_dest}:")
    for f in sorted(adapter_dest.iterdir()):
        print(f"  {f.name}  ({f.stat().st_size // 1024} KB)")

### Cell D5 — Upload DPO data files (skip if clone already brought them)

In [ ]:
from pathlib import Path

REQUIRED_FILES = [
    "data/explanations/dpo_train.jsonl",
    "data/explanations/dpo_val.jsonl",
]

missing = [f for f in REQUIRED_FILES if not Path(f).exists()]

if not missing:
    print("All DPO data files already present — skipping upload.")
    for f in REQUIRED_FILES:
        lines = sum(1 for _ in open(f))
        print(f"  {f}  ({lines} rows)")
else:
    print("Missing files — upload dpo_train.jsonl and dpo_val.jsonl:")
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        if "dpo_train" in name:
            dest = Path("data/explanations/dpo_train.jsonl")
        elif "dpo_val" in name:
            dest = Path("data/explanations/dpo_val.jsonl")
        else:
            print(f"Unexpected file: {name} — skipping")
            continue
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(data)
        print(f"  Saved {len(data)//1024} KB → {dest}")

### Cell D6 — Preflight: validate all DPO paths and print config

In [ ]:
import yaml
from pathlib import Path

cfg_path = Path("configs/phi3_dpo.yaml")
cfg = yaml.safe_load(cfg_path.read_text())

print("=" * 55)
print("configs/phi3_dpo.yaml")
print("=" * 55)
for k, v in cfg.items():
    print(f"  {k}: {v}")
print()

checks = {
    "preference_file" : cfg["preference_file"],
    "eval_file"       : cfg["eval_file"],
    "adapter_dir"     : cfg["adapter_dir"],
}
all_ok = True
print("Path checks:")
for label, path in checks.items():
    p = Path(path)
    if p.exists():
        if p.is_file():
            lines = sum(1 for _ in p.open())
            print(f"  OK  {label}: {path}  ({lines} rows)")
        else:
            files_in = list(p.iterdir())
            print(f"  OK  {label}: {path}  ({len(files_in)} files)")
    else:
        print(f"  MISSING  {label}: {path}")
        all_ok = False

Path(cfg["output_dir"]).parent.mkdir(parents=True, exist_ok=True)
print(f"  OK  output parent: {Path(cfg['output_dir']).parent}")

if not all_ok:
    raise FileNotFoundError(
        "Preflight failed.\n"
        "  - If adapter_dir is missing: re-run Cell D4 and upload phi3_veritas_qlora.zip\n"
        "  - If data files are missing: re-run Cell D5"
    )
print("\nPreflight passed. adapter_dir confirmed present.")

### Cell D7 — DPO dry-run

In [ ]:
import json
from pathlib import Path

!python3 scripts/train_phi3_dpo.py \
    --config configs/phi3_dpo.yaml \
    --dry-run \
    --report-json reports/phi3_dpo_dryrun.json \
    --report-md   reports/phi3_dpo_dryrun.md

report = json.loads(Path("reports/phi3_dpo_dryrun.json").read_text())
print("status:", report["status"])
print()
print("preflight_checks:")
all_ok = True
for c in report["preflight_checks"]:
    mark = "OK  " if c["exists"] else "FAIL"
    print(f"  [{mark}] {c['name']}: {c['path']}")
    if not c["exists"]:
        all_ok = False

if not all_ok:
    raise RuntimeError(
        "DPO dry-run failed.\n"
        "If adapter_exists is FAIL: Cell D4 did not place the adapter at adapters/phi3_veritas_qlora"
    )
print("\nAll checks passed. Ready to run DPO.")

### Cell D8 — Run DPO training

Expected time: **40–60 min** on a T4.  
DPO loads the QLoRA adapter as the policy model **and** a frozen reference model simultaneously — this is more memory-intensive than QLoRA. If you get CUDA OOM, see the recovery notes.

In [ ]:
!python3 scripts/train_phi3_dpo.py \
    --config configs/phi3_dpo.yaml \
    --report-json reports/phi3_dpo_training_metrics.json \
    --report-md   reports/phi3_dpo_training_metrics.md \
    --before-after-md reports/phi3_dpo_before_after_examples.md

### Cell D9 — Verify DPO adapter and print metrics

In [ ]:
import json
from pathlib import Path

report_path = Path("reports/phi3_dpo_training_metrics.json")
if not report_path.exists():
    raise FileNotFoundError("DPO training report missing — did Cell D8 finish without errors?")

report = json.loads(report_path.read_text())
print("status      :", report["status"])
print("output_dir  :", report["output_dir"])
print("adapter_dir :", report["adapter_dir"])

if report["status"] == "skipped":
    raise RuntimeError("DPO was skipped. Reason: " + report.get("reason", "unknown"))
if report["status"] != "trained":
    raise RuntimeError(f"Unexpected status: {report['status']}")

print("\ntrain_metrics:")
for k, v in report["train_metrics"].items():
    print(f"  {k}: {v}")

dpo_dir    = Path(report["output_dir"])
dpo_files  = sorted(dpo_dir.iterdir())
print(f"\nDPO adapter files ({dpo_dir}):")
for f in dpo_files:
    print(f"  {f.name}  ({f.stat().st_size // 1024} KB)")

if not any(f.name == "adapter_config.json" for f in dpo_files):
    raise FileNotFoundError("adapter_config.json missing.")
if not any("adapter_model" in f.name for f in dpo_files):
    raise FileNotFoundError("adapter_model weights missing.")
print("\nDPO adapter complete.")

### Cell D10 — Download DPO adapter and reports

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

dpo_dir  = Path("adapters/phi3_veritas_dpo")
zip_path = Path("/content/phi3_veritas_dpo.zip")

shutil.make_archive(
    str(zip_path.with_suffix("")), "zip",
    dpo_dir.parent, dpo_dir.name
)
print(f"Zipped: {zip_path}  ({zip_path.stat().st_size // 1024} KB)")
files.download(str(zip_path))

for rpt in [
    "reports/phi3_dpo_training_metrics.json",
    "reports/phi3_dpo_training_metrics.md",
    "reports/phi3_dpo_before_after_examples.md",
]:
    if Path(rpt).exists():
        files.download(rpt)
    else:
        print(f"  (missing: {rpt})")

---
## Recovery notes

**CUDA OOM during QLoRA**  
Reduce `max_seq_length` and `per_device_train_batch_size` before re-running Cell Q7:
```python
import yaml; from pathlib import Path
p = Path("configs/phi3_qlora.yaml")
cfg = yaml.safe_load(p.read_text())
cfg["max_seq_length"] = 1024
cfg["per_device_train_batch_size"] = 1
cfg["gradient_accumulation_steps"] = 16
p.write_text(yaml.dump(cfg))
print("Config updated.")
```

**CUDA OOM during DPO** (more likely — two models are loaded simultaneously)  
```python
import yaml; from pathlib import Path
p = Path("configs/phi3_dpo.yaml")
cfg = yaml.safe_load(p.read_text())
cfg["max_length"] = 1024
cfg["gradient_accumulation_steps"] = 32
p.write_text(yaml.dump(cfg))
print("Config updated.")
```

**Session disconnects mid-training**  
Reconnect, re-run cells Q1–Q4 (or D1–D4 for DPO), then check whether `adapters/phi3_veritas_qlora/` already has weights. If it does, skip training and go straight to verify + download. If session storage is gone, retrain from scratch.

**DPO dry-run shows `[FAIL] adapter_exists`**  
Check the zip structure before re-uploading:
```python
import zipfile
with zipfile.ZipFile("/content/phi3_veritas_qlora.zip") as zf:
    print(zf.namelist()[:8])  # first entry must start with phi3_veritas_qlora/
```

**Training report shows `status: skipped`**  
Read the `reason` field. Common causes: CUDA unavailable (wrong runtime), `bitsandbytes` not installed (forgot to restart after install), or for DPO, `adapter_dir` not found.